In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
print(device)

cuda


In [5]:
file_path_train = '/content/drive/My Drive/001.PhD/Smartphone/traintest/trainDataset2.csv'

In [6]:
df = pd.read_csv(file_path_train)

In [7]:
df.head()

,sequence
0,"android.net.nsd.STATE_CHANGED, android.net.con..."
1,"android.net.nsd.STATE_CHANGED, android.net.con..."
2,"android.intent.action.DREAMING_STARTED, androi..."
3,"android.net.wifi.RSSI_CHANGED, android.intent...."
4,"android.intent.action.DREAMING_STARTED, androi..."


In [8]:
sequence_list = df['sequence'].tolist()

In [9]:
sequence_list = sequence_list[:20000]
print(len(sequence_list))

20000


In [10]:
trainDataset = []
for sequence in sequence_list:
    result = sequence.split(",")
    result = [item.strip() for item in result]
    trainDataset.append(result)

In [ ]:
# print(trainDataset[0][-1])

In [ ]:
# trainDataset[0][0:-1]

In [11]:
from collections import Counter

In [12]:
all_tokens = [token for seq in trainDataset for token in seq]

In [13]:
counter = Counter(all_tokens)

In [14]:
# Vocabulary
vocab = {token: idx + 1 for idx, token in enumerate(counter.keys())}
vocab["<PAD>"] = 0

In [15]:
inv_vocab = {idx: token for token, idx in vocab.items()}
vocab_size = len(vocab)

In [16]:
from torch.nn.utils.rnn import pad_sequence

In [17]:
X = []
y = []

In [18]:
for seq in trainDataset:
    input_seq = [vocab[token] for token in seq[:-1]]
    target = vocab[seq[-1]]

    X.append(torch.tensor(input_seq))
    y.append(target)

In [19]:
# Pad sequences
X_padded = pad_sequence(X, batch_first=True, padding_value=0)
y = torch.tensor(y)

In [20]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_outputs):
        # lstm_outputs: (batch_size, seq_len, hidden_dim)

        energy = torch.tanh(self.attn(lstm_outputs))
        attention_scores = self.v(energy).squeeze(-1)
        attention_weights = F.softmax(attention_scores, dim=1)

        # Weighted sum of LSTM outputs
        context = torch.sum(
            lstm_outputs * attention_weights.unsqueeze(-1),
            dim=1
        )

        return context, attention_weights

In [21]:
class LSTMAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True
        )

        self.attention = Attention(hidden_dim)

        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_outputs, _ = self.lstm(embedded)

        context, attn_weights = self.attention(lstm_outputs)
        logits = self.fc(context)

        return logits, attn_weights

In [22]:
model = LSTMAttention(
    vocab_size=vocab_size,
    embed_dim=64,
    hidden_dim=128
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 170

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    logits, _ = model(X_padded)
    loss = criterion(logits, y)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 4.1145
Epoch 2, Loss: 4.0962
Epoch 3, Loss: 4.0781
Epoch 4, Loss: 4.0598
Epoch 5, Loss: 4.0409
Epoch 6, Loss: 4.0211
Epoch 7, Loss: 3.9998
Epoch 8, Loss: 3.9766
Epoch 9, Loss: 3.9509
Epoch 10, Loss: 3.9218
Epoch 11, Loss: 3.8883
Epoch 12, Loss: 3.8492
Epoch 13, Loss: 3.8029
Epoch 14, Loss: 3.7473
Epoch 15, Loss: 3.6795
Epoch 16, Loss: 3.5964
Epoch 17, Loss: 3.4939
Epoch 18, Loss: 3.3687
Epoch 19, Loss: 3.2182
Epoch 20, Loss: 3.0411
Epoch 21, Loss: 2.8456
Epoch 22, Loss: 2.6579
Epoch 23, Loss: 2.5182
Epoch 24, Loss: 2.4539
Epoch 25, Loss: 2.4468
Epoch 26, Loss: 2.4581
Epoch 27, Loss: 2.4638
Epoch 28, Loss: 2.4553
Epoch 29, Loss: 2.4310
Epoch 30, Loss: 2.3930
Epoch 31, Loss: 2.3450
Epoch 32, Loss: 2.2925
Epoch 33, Loss: 2.2403
Epoch 34, Loss: 2.1920
Epoch 35, Loss: 2.1511
Epoch 36, Loss: 2.1196
Epoch 37, Loss: 2.0958
Epoch 38, Loss: 2.0764
Epoch 39, Loss: 2.0584
Epoch 40, Loss: 2.0387
Epoch 41, Loss: 2.0165
Epoch 42, Loss: 1.9931
Epoch 43, Loss: 1.9702
Epoch 44, Loss: 1.94

In [23]:
file_path_test = '/content/drive/My Drive/001.PhD/Smartphone/traintest/testDataset2.csv'

In [24]:
dfTest = pd.read_csv(file_path_test)

In [25]:
dfTest.head()

,sequence
0,"android.net.wifi.RSSI_CHANGED, android.intent...."
1,"android.intent.action.DREAMING_STARTED, androi..."
2,"android.intent.action.DREAMING_STOPPED, androi..."
3,"android.net.wifi.RSSI_CHANGED, android.intent...."
4,"android.net.wifi.RSSI_CHANGED, android.intent...."


In [26]:
sequence_list_test = dfTest['sequence'].tolist()

In [27]:
sequence_list_test = sequence_list_test[:8000]

In [28]:
testDataset = []
for sequence in sequence_list_test:
    result = sequence.split(",")
    result = [item.strip() for item in result]
    testDataset.append(result)

In [29]:
X_test = []
y_test = []

In [30]:
for seq in testDataset:
    # Skip sequences with unseen tokens (optional but safe)
    if any(token not in vocab for token in seq):
        continue

    input_seq = [vocab[token] for token in seq[:-1]]
    target = vocab[seq[-1]]

    X_test.append(torch.tensor(input_seq))
    y_test.append(target)

In [31]:
X_test_padded = pad_sequence(X_test, batch_first=True, padding_value=0)
y_test = torch.tensor(y_test)

In [32]:
model.eval()

with torch.no_grad():
    logits, _ = model(X_test_padded)
    y_pred = torch.argmax(logits, dim=1)

In [33]:
y_true = y_test.cpu().numpy()
y_pred = y_pred.cpu().numpy()

In [34]:
from sklearn.metrics import precision_recall_fscore_support

In [35]:
precision, recall, _, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="micro",
    zero_division=0
)

_, _, f1_macro, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

_, _, f1_weighted, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

In [36]:
print(f"Precision    : {precision:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"F1 (macro)   : {f1_macro:.4f}")
print(f"F1 (weighted): {f1_weighted:.4f}")

Precision    : 0.7401
Recall       : 0.7401
F1 (macro)   : 0.1872
F1 (weighted): 0.7208
